# TP4 — Des arbres de décision aux méthodes d’ensemble

## Mission
Au moment où une commande Olist vient d’être validée et où la date estimée de livraison a été communiquée au client, peut-on **repérer les commandes présentant un risque de retard** ?

Nous reprenons **la même cible `is_late` et les 19 variables du TP3**. Notre objectif est de comprendre comment passer d’un arbre isolé à des modèles d’ensemble, puis de choisir un modèle pertinent pour Olist.

## Objectifs
- construire, visualiser et interpréter un arbre de décision ;
- comprendre et comparer **bagging** et **Random Forest** ;
- expérimenter le **Gradient Boosting** ;
- comparer **GBM, XGBoost, LightGBM et CatBoost** ;
- justifier une recommandation métier à partir des métriques, du temps de calcul et des limites de l’évaluation.

## 1. Préparation

In [ ]:
%pip -q install xgboost lightgbm catboost

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
pd.set_option("display.max_columns", 30)

In [ ]:
url = "https://raw.githubusercontent.com/ndiayemairame/miage-machine-learning/main/data/orders_ml.csv"
orders = pd.read_csv(url)
print("Dimensions :", orders.shape)
display(orders.head())

## 2. Retrouver le problème du TP3

L’unité d’observation est **une commande**. La cible `is_late` vaut 1 pour une commande livrée en retard, et 0 sinon. Le moment de prédiction reste **la validation de la commande, après communication de la date estimée**.

In [ ]:
display(orders["is_late"].value_counts())
print("Proportion de retards :", round(100 * orders["is_late"].mean(), 2), "%")

**Q1.** Pourquoi est-il utile de conserver exactement la même cible que dans le TP3 ?

**Q2.** Si les retards sont minoritaires, pourquoi l’accuracy peut-elle être trompeuse ?

**Q3.** Que représente un faux négatif pour Olist ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 3. Construire les jeux d’apprentissage, de validation et de test

Comme au TP3, nous utilisons une séparation **70 % / 15 % / 15 %**, stratifiée selon la cible. Pour que les quatre variantes de boosting restent exécutables dans Colab en séance, nous limitons **uniquement l’échantillon d’apprentissage** à 12 000 commandes, si nécessaire. Le jeu de validation et le test conservent leur définition initiale.

In [ ]:
train, temp = train_test_split(orders, test_size=0.30, random_state=RANDOM_STATE,
                               stratify=orders["is_late"])
validation, test = train_test_split(temp, test_size=0.50, random_state=RANDOM_STATE,
                                    stratify=temp["is_late"])

MAX_TRAIN = 12000
if len(train) > MAX_TRAIN:
    train_tp, _ = train_test_split(train, train_size=MAX_TRAIN,
                                   stratify=train["is_late"], random_state=RANDOM_STATE)
else:
    train_tp = train.copy()

for name, df in [("apprentissage TP4", train_tp), ("validation", validation), ("test", test)]:
    print(name, len(df), "commandes —", round(100 * df["is_late"].mean(), 2), "% de retards")

In [ ]:
features = [
    "promised_delivery_days", "purchase_month", "purchase_day_of_week", "purchase_hour",
    "n_items", "n_products", "n_sellers", "total_price", "total_freight",
    "mean_product_weight_g", "max_product_weight_g",
    "mean_product_volume_cm3", "max_product_volume_cm3", "n_categories",
    "freight_ratio", "seller_customer_distance_mean", "seller_customer_distance_max",
    "same_state_share", "payment_installments_max"
]
target = "is_late"
X_train, y_train = train_tp[features], train_tp[target]
X_val, y_val = validation[features], validation[target]
X_test, y_test = test[features], test[target]

imputer = SimpleImputer(strategy="median")
X_train_p = pd.DataFrame(imputer.fit_transform(X_train), columns=features, index=X_train.index)
X_val_p = pd.DataFrame(imputer.transform(X_val), columns=features, index=X_val.index)
X_test_p = pd.DataFrame(imputer.transform(X_test), columns=features, index=X_test.index)
print("Variables explicatives :", len(features))

**Q4.** Pourquoi faut-il ajuster l’imputation uniquement sur l’apprentissage ?

**Q5.** Pourquoi `delivery_time_days`, `carrier_time_days` et `review_score` sont-elles exclues ?

**Q6.** Quel compromis introduit le sous-échantillonnage du jeu d’apprentissage ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 4. Premier arbre de décision

Un arbre de décision construit des règles successives à partir des variables explicatives.


In [ ]:
tree_small = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50,
                                    random_state=RANDOM_STATE)
tree_small.fit(X_train_p, y_train)
plt.figure(figsize=(15, 5))
plot_tree(tree_small, feature_names=features, class_names=["À l'heure", "En retard"],
          filled=True, rounded=True, fontsize=8, max_depth=3)
plt.title("Arbre de décision — profondeur maximale 3")
plt.show()

**Q7.** Quelle variable intervient dans la première séparation ? Que signifie le seuil affiché ?

**Q8.** Choisissez une feuille : comment interpréter la proportion de commandes en retard qui y figurent ?

**Q9.** Un arbre facile à expliquer est-il nécessairement un bon prédicteur ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 5. Que se passe-t-il quand on complexifie l’arbre ?

Nous entraînons plusieurs arbres en faisant varier leur profondeur. Comparez les scores sur les données utilisées pour l’entraînement et sur des commandes non utilisées pour cet entraînement.


In [ ]:
def score_model(model, X_fit=X_train_p, y_fit=y_train, X_eval=X_val_p, y_eval=y_val):
    start = time.perf_counter()
    model.fit(X_fit, y_fit)
    duration = time.perf_counter() - start
    prob = model.predict_proba(X_eval)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        "ROC-AUC": roc_auc_score(y_eval, prob),
        "PR-AUC": average_precision_score(y_eval, prob),
        "Rappel (0.5)": recall_score(y_eval, pred, zero_division=0),
        "Précision (0.5)": precision_score(y_eval, pred, zero_division=0),
        "F1 (0.5)": f1_score(y_eval, pred, zero_division=0),
        "Temps (s)": duration
    }

depths = [2, 3, 5, 8, 12, None]
rows = []
for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=5,
                                   random_state=RANDOM_STATE)
    model.fit(X_train_p, y_train)
    rows.append({"profondeur": str(depth),
                 "ROC-AUC train": roc_auc_score(y_train, model.predict_proba(X_train_p)[:, 1]),
                 "ROC-AUC validation": roc_auc_score(y_val, model.predict_proba(X_val_p)[:, 1])})
depth_results = pd.DataFrame(rows)
display(depth_results)
ax = depth_results.plot(x="profondeur", y=["ROC-AUC train", "ROC-AUC validation"],
                        marker="o", figsize=(8, 4))
ax.set_ylabel("ROC-AUC")
ax.set_title("Complexité et généralisation d'un arbre")
plt.show()

**Q10.** Que se passe-t-il sur l’apprentissage quand la profondeur augmente ?

**Q11.** À partir de quel moment constatez-vous un écart entre apprentissage et validation ?

**Q12.** Quel lien faites-vous avec le compromis biais-variance ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 6. Expérience : plusieurs arbres sur des échantillons différents

Nous allons comparer un arbre seul à un ensemble d'arbres entraînés sur des tirages aléatoires des commandes d'apprentissage. Chaque tirage est réalisé **avec remise** : une commande peut apparaître plusieurs fois dans un tirage et ne pas apparaître dans un autre.

Cette technique est appelée **bagging** (*bootstrap aggregating*).


In [ ]:
single_tree = DecisionTreeClassifier(min_samples_leaf=5, random_state=RANDOM_STATE)
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(min_samples_leaf=5),
    n_estimators=60, bootstrap=True, n_jobs=-1, random_state=RANDOM_STATE
)
results = {}
results["Arbre"] = score_model(single_tree)
results["Bagging"] = score_model(bagging)
display(pd.DataFrame(results).T.round(3))

**Q13.** Comment les échantillons utilisés par les différents arbres ont-ils été construits ?

**Q14.** Comparez les résultats de l'arbre seul et du bagging. Qu'observez-vous ?

**Q15.** À votre avis, qu'apporte l'agrégation de plusieurs prédictions ? Appuyez-vous sur les résultats.


### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 7. Une autre façon de construire une forêt

Nous testons maintenant une **Random Forest**. Lorsqu'un arbre cherche une séparation, il n'examine qu'un sous-ensemble aléatoire des variables candidates.

In [ ]:
forest = RandomForestClassifier(n_estimators=120, min_samples_leaf=5,
                                max_features="sqrt", n_jobs=-1,
                                random_state=RANDOM_STATE)
results["Random Forest"] = score_model(forest)
display(pd.DataFrame(results).T.round(3))

**Q16.** En examinant les paramètres et la description des expériences, quelle différence repérez-vous entre le bagging précédent et la Random Forest ?

**Q17.** Que montrent les résultats de validation ? La même méthode domine-t-elle sur tous les critères ?

**Q18.** Quels résultats faudrait-il regarder pour savoir si les modèles se généralisent bien ?


### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 8. Une autre stratégie d'ensemble : le Gradient Boosting

Nous passons à une nouvelle famille de modèles. Dans un **Gradient Boosting Machine (GBM)**, les arbres sont ajoutés **successivement** afin de réduire progressivement une fonction de perte.

Les paramètres `n_estimators` et `learning_rate` contrôlent respectivement le nombre d'étapes et la contribution de chacune.


In [ ]:
gbm = GradientBoostingClassifier(n_estimators=100, learning_rate=0.08,
                                 max_depth=2, random_state=RANDOM_STATE)
results["GBM classique"] = score_model(gbm)
display(pd.DataFrame(results).T.round(3))

**Q19.** Comment les arbres sont-ils construits dans le GBM ? Comparez cette organisation à celle du bagging.

**Q20.** Comparez les résultats du GBM à ceux des ensembles précédents. Qu'en déduisez-vous pour ces données ?

**Q21.** Selon vous, quels risques peut-on rencontrer en augmentant fortement le nombre d'arbres ? Comment le vérifier expérimentalement ?


### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 9. Faire varier le taux d’apprentissage

Testez plusieurs valeurs de `learning_rate` en conservant les autres paramètres identiques.


In [ ]:
gbm_experiments = []
for lr in [0.03, 0.10, 0.30]:
    model = GradientBoostingClassifier(n_estimators=80, learning_rate=lr,
                                       max_depth=2, random_state=RANDOM_STATE)
    scores = score_model(model)
    gbm_experiments.append({"learning_rate": lr, **scores})
display(pd.DataFrame(gbm_experiments).round(3))

**Q22.** Quelle valeur de `learning_rate` donne la meilleure PR-AUC sur validation dans cette expérience ?

**Q23.** Peut-on conclure qu’elle sera toujours optimale ? Pourquoi ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 10. Trois implémentations modernes du boosting

Le **Gradient Boosting** désigne une famille de méthodes, pas une seule bibliothèque. Nous allons tester trois implémentations qui construisent elles aussi des ensembles d'arbres successifs, mais avec des choix de conception différents. Pour plus d'informations sur ces méthodes, consultez le polycopié de cours.

### XGBoost — *Extreme Gradient Boosting*

XGBoost optimise une fonction de perte en ajoutant des arbres successifs. Il utilise notamment des informations issues des **dérivées de la perte** pour choisir les améliorations à apporter. Il intègre des mécanismes de **régularisation** et peut échantillonner les observations et les variables pour limiter la complexité et accélérer certains calculs.

### LightGBM — *Light Gradient Boosting Machine*

LightGBM vise un entraînement efficace, notamment sur de grands jeux de données. Il regroupe les valeurs numériques en **histogrammes** pour accélérer la recherche de séparations. Sa croissance des arbres privilégie généralement la **feuille dont la séparation apporte le plus grand gain**, plutôt qu'une croissance uniforme par niveau.

### CatBoost — *Categorical Boosting*

CatBoost a été conçu notamment pour faciliter l'utilisation de **variables catégorielles**. Il propose des traitements spécifiques de ces variables et des mécanismes d'apprentissage destinés à limiter certains biais liés à leur encodage. Il utilise aussi, par défaut, des **arbres symétriques**, dont les séparations suivent une structure régulière.

### Notre protocole de comparaison

Pour comparer les algorithmes sans changer simultanément la représentation des données, nous leur donnons ici **les mêmes 19 variables numériques préparées** et les mêmes ensembles d'apprentissage et de validation. Nous **ne testons pas** dans cette expérience l'avantage éventuel de CatBoost sur les catégories brutes.

Les paramètres choisis sont des réglages de départ, **pas une optimisation exhaustive**.


In [ ]:
boosting_models = {
    "XGBoost": XGBClassifier(n_estimators=120, max_depth=3, learning_rate=0.08,
                              subsample=0.9, colsample_bytree=0.9,
                              eval_metric="logloss", n_jobs=2, random_state=RANDOM_STATE),
    "LightGBM": LGBMClassifier(n_estimators=120, max_depth=4, num_leaves=15,
                                learning_rate=0.08, verbosity=-1,
                                n_jobs=2, random_state=RANDOM_STATE),
    "CatBoost": CatBoostClassifier(iterations=120, depth=4, learning_rate=0.08,
                                    verbose=False, thread_count=2,
                                    random_seed=RANDOM_STATE)
}
for name, model in boosting_models.items():
    results[name] = score_model(model)
    print(name, "terminé")

display(pd.DataFrame(results).T.round(3))

**Q24.** Classez les quatre implémentations de boosting selon leur PR-AUC sur validation. Que constatez-vous ?

**Q25.** Refaites ce classement selon le temps d'entraînement. Est-il identique ?

**Q26.** À partir des descriptions des trois bibliothèques, choisissez **une particularité technique de chacune** et expliquez dans quel contexte elle pourrait être utile.

**Q27.** Que faudrait-il modifier dans le protocole pour étudier spécifiquement la gestion des variables catégorielles par CatBoost ?


### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 11. Mettre les résultats en regard

Regroupez les résultats de tous les modèles. Comparez les métriques de classement (`ROC-AUC`, `PR-AUC`), les métriques calculées au seuil de 0.5 (précision, rappel, F1) et le temps d'entraînement.

In [ ]:
comparison = pd.DataFrame(results).T.sort_values("PR-AUC", ascending=False)
display(comparison.round(3))

comparison[["ROC-AUC", "PR-AUC"]].plot(kind="bar", figsize=(11, 4), rot=35)
plt.title("Comparaison des modèles sur validation")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

**Q28.** Le modèle le plus performant en ROC-AUC est-il aussi le meilleur en PR-AUC ?

**Q29.** Quel modèle retiendriez-vous si le temps de calcul comptait beaucoup ?

**Q30.** Une différence très faible entre deux scores suffit-elle à conclure que l’un des modèles est meilleur ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 12. Quelles variables les modèles utilisent-ils ?

Les modèles d'arbres fournissent des indicateurs d'importance des variables. Nous allons comparer ceux de Random Forest et du GBM.


In [ ]:
importance = pd.DataFrame({
    "variable": features,
    "Random Forest": forest.feature_importances_,
    "GBM": gbm.feature_importances_
}).set_index("variable")
importance.sort_values("GBM").tail(10).plot(kind="barh", figsize=(9, 5))
plt.title("Importance des variables : deux modèles")
plt.tight_layout()
plt.show()

**Q31.** Quelles variables semblent les plus influentes dans les deux modèles ?

**Q32.** Pourquoi les classements d’importance peuvent-ils différer ?

**Q33.** Peut-on conclure qu’augmenter une variable importante *cause* un retard ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 13. Passer des probabilités à une décision

Chez Olist, chaque alerte sur une commande potentiellement en retard peut déclencher une action opérationnelle. À partir du modèle retenu sur validation, faisons varier le seuil de décision et observons le nombre d'alertes ainsi que les erreurs commises.


In [ ]:
best_name = comparison.index[0]
all_models = {"Arbre": single_tree, "Bagging": bagging, "Random Forest": forest,
              "GBM classique": gbm, **boosting_models}
best_model = all_models[best_name]
proba_val = best_model.predict_proba(X_val_p)[:, 1]
threshold_rows = []
for threshold in [0.10, 0.20, 0.30, 0.50]:
    pred = (proba_val >= threshold).astype(int)
    threshold_rows.append({"seuil": threshold,
                           "precision": precision_score(y_val, pred, zero_division=0),
                           "rappel": recall_score(y_val, pred, zero_division=0),
                           "f1": f1_score(y_val, pred, zero_division=0),
                           "alertes": int(pred.sum())})
print("Modèle étudié :", best_name)
display(pd.DataFrame(threshold_rows).round(3))

**Q34.** Comment évoluent le rappel et le nombre d’alertes quand le seuil diminue ?

**Q35.** Quel seuil recommanderiez-vous si manquer un retard coûte cher ? Quelle information métier manque-t-il pour décider définitivement ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 14. Évaluation finale : à vous de jouer

**Choisissez maintenant un modèle et un seuil à partir des résultats de validation.** Évaluez-les ensuite **une seule fois** sur le jeu de test, sans changer votre choix à la vue des résultats.

Pour cette séance, nous conservons le modèle entraîné sur `train_tp` afin d’isoler l’évaluation finale. Dans un projet complet, on pourrait ensuite réentraîner le modèle sélectionné sur apprentissage + validation, en réajustant toute la préparation des données **sans utiliser le test**.

In [ ]:
# À VOUS DE JOUER

# 1. Indiquez le modèle et le seuil choisis
modele_choisi = "..."
seuil_choisi = ...

# 2. Récupération du modèle
modele_final = all_models[modele_choisi]

# 3. Prédictions sur le jeu de test
proba_test = ...
pred_test = ...

# 4. Évaluation finale
print("ROC-AUC :", round(roc_auc_score(y_test, proba_test), 3))
print("PR-AUC :", round(average_precision_score(y_test, proba_test), 3))
print("Précision :", round(precision_score(y_test, pred_test, zero_division=0), 3))
print("Rappel :", round(recall_score(y_test, pred_test, zero_division=0), 3))

# 5. Matrice de confusion
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_test,
    display_labels=["À l'heure", "En retard"],
    values_format="d"
)
plt.title("Matrice de confusion — jeu de test")
plt.show()

**Q36.** Quel modèle et quel seuil avez-vous choisis ?

**Q37.** Les résultats sur test sont-ils proches de ceux observés sur validation ?

**Q38.** Pourquoi ne faut-il pas modifier le seuil ou les hyperparamètres après avoir consulté les résultats sur test ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

# 15. Bilan — Votre recommandation à Olist

**Q39.** Rédigez une recommandation de 5 à 8 lignes pour l’équipe opérationnelle d’Olist. Elle doit inclure :

- le modèle retenu et son intérêt par rapport à un arbre seul ;
- la métrique principale utilisée pour le sélectionner ;
- le seuil et ses conséquences en matière d’alertes ;
- au moins une limite de votre étude.

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

---

## À retenir

Ce TP montre comment passer d’un arbre de décision à des ensembles d’arbres, puis comment comparer ces méthodes sur un même problème métier : anticiper les retards de livraison chez Olist.

- Arbre de décision : simple à interpréter, mais sensible au surapprentissage.
- Bagging / Random Forest : combinent plusieurs arbres pour stabiliser les prédictions.
- Boosting (GBM, XGBoost, LightGBM, CatBoost) : construit les modèles successivement pour améliorer la prédiction.

Idée essentielle : un modèle plus complexe n’est pas automatiquement un meilleur choix. La décision doit reposer sur des résultats mesurés sur des données non vues et sur les besoins opérationnels de l'entreprise.
